# Replace a model
Downloads a model from Hugging Face and installs it on Google Drive under a pipeline name, so the cloud agent uses it from its next session.

1. Stop the cloud agent.
2. Fill in **2a** below.
3. **Runtime → Run all**.

See *How to replace models* in `MODELS_GUIDE.md`.

In [5]:
# ==============================================================================
# 1. INSTALLS & IMPORTS
# ==============================================================================
import re, shutil, datetime
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download
from huggingface_hub.utils import GatedRepoError, RepositoryNotFoundError

try:
    from google.colab import drive      # present in Colab only
except ImportError:
    drive = None

In [6]:
# ==============================================================================
# 2. CONFIGURATION
#    Change the settings in 2a, then run all cells. Nothing else needs editing.
# ==============================================================================

# ------------------------------------------------------------------------------
# 2a. What to do
# ------------------------------------------------------------------------------
ROLE = "local-analyst"         # "local-analyst" = the writer (language model); "local-embed" = the sorter (embeddings model)
ACTION = "replace"             # "replace" = install a new model; "rollback" = go back to the previous one;
                               # "delete_previous" = remove the previous one to free Drive space

REPO_ID = "bartowski/Mistral-Small-24B-Instruct-2501-GGUF"   # the address at the top of the model's Hugging Face page
FILENAME = ""                  # exact file name from "Files and versions"; "" = pick the QUANT file automatically
QUANT = "Q4_K_M"               # the compression level to pick when FILENAME is blank
KEEP_PREVIOUS = True           # keep the old model as <role>.previous.gguf, so you can roll back

# ------------------------------------------------------------------------------
# 2b. Size limits: the largest file each role may be, in GB (a T4 GPU has ~15 GB)
# ------------------------------------------------------------------------------
MAX_FILE_GB = {"local-analyst": 12.5, "local-embed": 1.5}
ALLOW_OVERSIZE = True        # True = install anyway (it will run slowly if it doesn't fit in GPU memory)

# ------------------------------------------------------------------------------
# 2c. Folders (the same as the other notebooks)
# ------------------------------------------------------------------------------
DRIVE_MOUNT_POINT = "/content/drive"
DIRECTORY_NAME = "PhD-research-assistant"
DRIVE_DIR = Path(DRIVE_MOUNT_POINT) / "MyDrive" / DIRECTORY_NAME
DRIVE_MODELS = DRIVE_DIR / "models"
LOCAL_MODELS = Path("/content/phdresearch-models")          # Colab's own disk: the download lands here first
MODEL_SOURCES_FILE = DRIVE_MODELS / "MODEL_SOURCES.md"      # a record of which model is behind each name

ROLES = {"local-analyst": "the writer (language model)", "local-embed": "the sorter (embeddings model)"}
MODEL_SUFFIX = ".gguf"
PREVIOUS_SUFFIX = ".previous.gguf"
PART_SUFFIX = ".part"

In [7]:
# ==============================================================================
# 3. FUNCTIONS
# ==============================================================================
def gb(size_bytes: int) -> float:
    return size_bytes / 1e9

def model_paths(role: str):
    """(current, previous) model files on Drive for a role."""
    return DRIVE_MODELS / f"{role}{MODEL_SUFFIX}", DRIVE_MODELS / f"{role}{PREVIOUS_SUFFIX}"

def find_model_file(repo_id: str, filename: str, quant: str):
    """Returns (file name, size in bytes) of the GGUF file to download, or stops with a plain explanation."""
    try:
        info = HfApi().model_info(repo_id, files_metadata=True)
    except GatedRepoError:
        raise SystemExit(f"[!] {repo_id} asks you to log in and accept its terms. Choose a different model.")
    except RepositoryNotFoundError:
        raise SystemExit(f"[!] No model found at '{repo_id}'. Copy the address from the top of the model's page.")

    files = {s.rfilename: s.size for s in info.siblings
             if s.rfilename.lower().endswith(".gguf") and "mmproj" not in s.rfilename.lower()}   # mmproj = image add-on
    if not files:
        raise SystemExit(f"[!] {repo_id} has no GGUF files. Look for a version whose name ends in GGUF.")

    split = re.compile(r"-\d{5}-of-\d{5}\.gguf$", re.IGNORECASE)          # models too big for one file
    if filename:
        if filename not in files:
            raise SystemExit(f"[!] '{filename}' isn't in {repo_id}. GGUF files there:\n    " + "\n    ".join(sorted(files)))
        chosen = filename
    else:
        matches = sorted(f for f in files if quant.lower() in f.lower() and not split.search(f))
        if not matches and any(quant.lower() in f.lower() for f in files):
            raise SystemExit(f"[!] The {quant} version of this model is split into several parts, which this project can't use. "
                             "It's too big anyway: choose a smaller model.")
        if not matches:
            raise SystemExit(f"[!] No {quant} file in {repo_id}. Set FILENAME to one of:\n    " + "\n    ".join(sorted(files)))
        chosen = min(matches, key=len)                                    # the plainest name if there are several
    if split.search(chosen):
        raise SystemExit(f"[!] '{chosen}' is split into several parts, which this project can't use. "
                         "Choose a smaller model or a more compressed file.")
    return chosen, files[chosen]

def check_size(role: str, size: int) -> None:
    limit = MAX_FILE_GB[role]
    print(f"[i] File size: {gb(size):.1f} GB (limit for {role}: {limit} GB)")
    if gb(size) > limit:
        message = (f"[!] This file is bigger than {limit} GB, so it may not fit in GPU memory and would run slowly. "
                   "Choose a smaller model or a more compressed file (see the Models guide).")
        if not ALLOW_OVERSIZE: raise SystemExit(message + " To install it anyway, set ALLOW_OVERSIZE = True.")
        print(message + " Installing anyway (ALLOW_OVERSIZE = True).")
    local_free = shutil.disk_usage(LOCAL_MODELS).free
    if local_free < size + 1e9:
        raise SystemExit(f"[!] Colab's disk has {gb(local_free):.1f} GB free; this download needs {gb(size) + 1:.1f} GB.")
    try:
        drive_free = shutil.disk_usage(DRIVE_MODELS).free
        if drive_free < size * 1.05:
            print(f"[!] Warning: your Google Drive may only have {gb(drive_free):.1f} GB free. "
                  "If the copy to Drive fails, free up space or set KEEP_PREVIOUS = False.")
    except OSError:
        pass

def download(repo_id: str, filename: str, size: int) -> Path:
    print(f"[+] Downloading {filename} to Colab's disk ...")
    path = Path(hf_hub_download(repo_id=repo_id, filename=filename, local_dir=str(LOCAL_MODELS)))
    with open(path, "rb") as f:
        if f.read(4) != b"GGUF":
            raise SystemExit("[!] The downloaded file isn't a valid GGUF model. Nothing on Drive was changed.")
    if path.stat().st_size != size:
        raise SystemExit("[!] The download is incomplete. Run this cell again. Nothing on Drive was changed.")
    return path

def install(role: str, downloaded: Path) -> None:
    """Copies to Drive as .part, then swaps it in, so a failed copy never replaces a working model."""
    current, previous = model_paths(role)
    part = current.with_name(current.name + PART_SUFFIX)
    print("[+] Copying to Google Drive (this can take several minutes for a large model) ...")
    shutil.copy2(downloaded, part)
    if current.exists():
        if KEEP_PREVIOUS:
            if previous.exists(): previous.unlink()
            current.rename(previous)
            print(f"[i] The old model is kept as {previous.name}")
        else:
            current.unlink()
    part.rename(current)
    downloaded.unlink()
    shutil.rmtree(LOCAL_MODELS / ".cache", ignore_errors=True)       # the download tool's working files
    print(f"[✓] {current.name} is now {downloaded.name}")

def record_source(role: str, text: str) -> None:
    """Appends a dated line to MODEL_SOURCES.md, so you can always see which model is behind each name."""
    stamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
    new_file = not MODEL_SOURCES_FILE.exists()
    with open(MODEL_SOURCES_FILE, "a", encoding="utf-8") as f:
        if new_file: f.write("# Model sources\n\nWhich model is installed behind each pipeline name, newest last.\n\n")
        f.write(f"- {stamp} | **{role}** | {text}\n")

def replace_model(role: str) -> None:
    filename, size = find_model_file(REPO_ID, FILENAME, QUANT)
    print(f"[i] Replacing {role} ({ROLES[role]}) with {REPO_ID} / {filename}")
    check_size(role, size)
    downloaded = download(REPO_ID, filename, size)
    install(role, downloaded)
    record_source(role, f"installed `{REPO_ID}` / `{filename}` ({gb(size):.1f} GB)")
    print("\n[i] To make a fresh install download the same model, set this entry in REQUIRED_MODELS in agenticSetup.ipynb:")
    print(f'    "{role}{MODEL_SUFFIX}": {{\n        "repo_id": "{REPO_ID}",\n        "filename": "{filename}"\n    }},')

def rollback(role: str) -> None:
    current, previous = model_paths(role)
    if not previous.exists():
        raise SystemExit(f"[!] There is no previous model for {role} to go back to.")
    swap = current.with_name(current.name + ".swap")
    if current.exists(): current.rename(swap)
    previous.rename(current)
    if swap.exists(): swap.rename(previous)                           # the model you rolled back from becomes "previous"
    record_source(role, "rolled back to the previous model")
    print(f"[✓] {role} is back on the previous model. The one you rolled back from is kept as {previous.name}.")

def delete_previous(role: str) -> None:
    _, previous = model_paths(role)
    if not previous.exists():
        print(f"[i] There is no previous model for {role}. Nothing to delete.")
        return
    size = previous.stat().st_size
    previous.unlink()
    record_source(role, "deleted the previous model")
    print(f"[✓] Deleted {previous.name}, freeing {gb(size):.1f} GB on Drive.")

In [8]:
# ==============================================================================
# 4. RUN
#    Stop the cloud agent before running this. The new model is used from its next session.
# ==============================================================================
if ROLE not in ROLES:
    raise SystemExit(f"[!] ROLE must be one of: {', '.join(ROLES)}")
if drive is not None:
    drive.mount(DRIVE_MOUNT_POINT)
for p in (DRIVE_MODELS, LOCAL_MODELS): p.mkdir(parents=True, exist_ok=True)

actions = {"replace": replace_model, "rollback": rollback, "delete_previous": delete_previous}
if ACTION not in actions:
    raise SystemExit(f"[!] ACTION must be one of: {', '.join(actions)}")
actions[ACTION](ROLE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[i] Replacing local-analyst (the writer (language model)) with bartowski/Mistral-Small-24B-Instruct-2501-GGUF / Mistral-Small-24B-Instruct-2501-Q4_K_M.gguf
[i] File size: 14.3 GB (limit for local-analyst: 12.5 GB)
[!] This file is bigger than 12.5 GB, so it may not fit in GPU memory and would run slowly. Choose a smaller model or a more compressed file (see the Models guide). Installing anyway (ALLOW_OVERSIZE = True).
[+] Downloading Mistral-Small-24B-Instruct-2501-Q4_K_M.gguf to Colab's disk ...


Mistral-Small-24B-Instruct-2501-Q4_K_M.g(…): reconstructing file:   0%|          |  0.00B / 14.3GB            

Mistral-Small-24B-Instruct-2501-Q4_K_M.g(…): downloading bytes:           |  0.00B            

[+] Copying to Google Drive (this can take several minutes for a large model) ...
[✓] local-analyst.gguf is now Mistral-Small-24B-Instruct-2501-Q4_K_M.gguf

[i] To make a fresh install download the same model, set this entry in REQUIRED_MODELS in agenticSetup.ipynb:
    "local-analyst.gguf": {
        "repo_id": "bartowski/Mistral-Small-24B-Instruct-2501-GGUF",
        "filename": "Mistral-Small-24B-Instruct-2501-Q4_K_M.gguf"
    },
